Reconciling 2017 dataset

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir_2017 = Path("../data/raw/cicids17")
data_dir_2018 = Path("../data/raw/cicids18")

files_2017 = sorted(data_dir_2017.glob("*.csv"))
files_2018 = sorted(data_dir_2018.glob("*.parquet"))

print("CIC-IDS2017 files:", len(files_2017))
print("CSE-CIC-IDS2018 files:", len(files_2018))

CIC-IDS2017 files: 8
CSE-CIC-IDS2018 files: 10


In [2]:
common_feature_mapping = {
    "Average Packet Size": "Avg Packet Size",
    "Total Length of Bwd Packets": "Bwd Packets Length Total",
    "Total Length of Fwd Packets": "Fwd Packets Length Total",
    "act_data_pkt_fwd": "Fwd Act Data Packets",
    "min_seg_size_forward": "Fwd Seg Size Min",
    "Init_Win_bytes_backward": "Init Bwd Win Bytes",
    "Init_Win_bytes_forward": "Init Fwd Win Bytes",
    "Max Packet Length": "Packet Length Max",
    "Min Packet Length": "Packet Length Min"
}

excluded_features_2017 = [
    "Destination Port",
    "Fwd Header Length.1",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

excluded_features_2018 = [
    "Protocol",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

print("Feature mappings:", len(common_feature_mapping))
print("2017 exclusions:", len(excluded_features_2017))
print("2018 exclusions:", len(excluded_features_2018))

Feature mappings: 9
2017 exclusions: 10
2018 exclusions: 9


In [3]:
sample_2017 = pd.read_csv(files_2017[0], nrows=5)
sample_2017.columns = sample_2017.columns.str.strip()

sample_2018 = pd.read_parquet(files_2018[0]).head(5)

features_2017 = [
    col for col in sample_2017.columns
    if col != "Label" and col not in excluded_features_2017
]

features_2018 = [
    col for col in sample_2018.columns
    if col != "Label" and col not in excluded_features_2018
]

features_2017_reconciled = [
    common_feature_mapping.get(col, col)
    for col in features_2017
]

print("2017 features:", len(features_2017_reconciled))
print("2018 features:", len(features_2018))
print()
print("Same feature set:", set(features_2017_reconciled) == set(features_2018))

2017 features: 68
2018 features: 68

Same feature set: True


In [5]:
print("Number of common modeling features:", len(features_2017_reconciled))
print()
print(features_2017_reconciled)

Number of common modeling features: 68

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 

In [6]:
def preprocess_2017_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns=common_feature_mapping)

    df = df[features_2017_reconciled + ["Label"]].copy()

    df["Original_Label"] = df["Label"].astype(str).str.strip()
    df["Target"] = (
        df["Original_Label"].str.upper() != "BENIGN"
    ).astype(np.int8)

    df["Source_File"] = file_path.name

    df = df.drop(columns=["Label"])

    df[features_2017_reconciled] = (
        df[features_2017_reconciled]
        .replace([np.inf, -np.inf], np.nan)
    )

    expected_columns = (
        features_2017_reconciled
        + ["Original_Label", "Target", "Source_File"]
    )

    assert list(df.columns) == expected_columns
    assert len(features_2017_reconciled) == 68
    assert df["Target"].isin([0, 1]).all()

    return df

In [7]:
test_df_2017 = preprocess_2017_file(files_2017[0])

print("Shape:", test_df_2017.shape)
print()
print("Feature columns:", len(features_2017_reconciled))
print("Target values:", sorted(test_df_2017["Target"].unique()))
print()
print("Missing values:", test_df_2017[features_2017_reconciled].isna().sum().sum())
print("Infinite values:", np.isinf(test_df_2017[features_2017_reconciled].to_numpy()).sum())
print()
print("Original labels:")
print(test_df_2017["Original_Label"].value_counts())

Shape: (225745, 71)

Feature columns: 68
Target values: [np.int8(0), np.int8(1)]

Missing values: 68
Infinite values: 0

Original labels:
Original_Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


Reconciling 2018 dataset

In [8]:
def preprocess_2018_file(file_path):
    df = pd.read_parquet(file_path)

    df = df[features_2018].copy()

    label_column = "Label"

    original_label = pd.read_parquet(
        file_path,
        columns=[label_column]
    )[label_column].astype(str).str.strip()

    df["Original_Label"] = original_label
    df["Target"] = (
        df["Original_Label"].str.upper() != "BENIGN"
    ).astype(np.int8)

    df["Source_File"] = file_path.name

    df[features_2018] = (
        df[features_2018]
        .replace([np.inf, -np.inf], np.nan)
    )

    expected_columns = (
        features_2018
        + ["Original_Label", "Target", "Source_File"]
    )

    assert list(df.columns) == expected_columns
    assert len(features_2018) == 68
    assert df["Target"].isin([0, 1]).all()

    return df

In [9]:
test_df_2018 = preprocess_2018_file(files_2018[0])

print("Shape:", test_df_2018.shape)
print()
print("Feature columns:", len(features_2018))
print("Target values:", sorted(test_df_2018["Target"].unique()))
print()
print("Missing values:", test_df_2018[features_2018].isna().sum().sum())
print("Infinite values:", np.isinf(test_df_2018[features_2018].to_numpy()).sum())
print()
print("Original labels:")
print(test_df_2018["Original_Label"].value_counts())

Shape: (771587, 71)

Feature columns: 68
Target values: [np.int8(0), np.int8(1)]

Missing values: 0
Infinite values: 0

Original labels:
Original_Label
Benign    627052
Bot       144535
Name: count, dtype: int64


In [10]:
processed_dir_2017 = Path("../data/processed/cicids17")
processed_dir_2018 = Path("../data/processed/cicids18")

processed_dir_2017.mkdir(parents=True, exist_ok=True)
processed_dir_2018.mkdir(parents=True, exist_ok=True)

print("Processed directories ready.")

Processed directories ready.


In [11]:
processed_counts_2017 = []

for file_path in files_2017:
    output_path = processed_dir_2017 / f"{file_path.stem}.parquet"

    df = preprocess_2017_file(file_path)
    df.to_parquet(output_path, index=False)

    processed_counts_2017.append({
        "file": file_path.name,
        "rows": len(df),
        "missing": df[features_2017_reconciled].isna().sum().sum()
    })

    del df

processed_counts_2017 = pd.DataFrame(processed_counts_2017)

print(processed_counts_2017)
print()
print("Total rows:", processed_counts_2017["rows"].sum())
print("Total missing:", processed_counts_2017["missing"].sum())

                                                file    rows  missing
0   Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  225745       68
1  Friday-WorkingHours-Afternoon-PortScan.pcap_IS...  286467      742
2          Friday-WorkingHours-Morning.pcap_ISCX.csv  191033      244
3                  Monday-WorkingHours.pcap_ISCX.csv  529918      874
4  Thursday-WorkingHours-Afternoon-Infilteration....  288602      414
5  Thursday-WorkingHours-Morning-WebAttacks.pcap_...  170366      270
6                 Tuesday-WorkingHours.pcap_ISCX.csv  445909      528
7               Wednesday-workingHours.pcap_ISCX.csv  692703     2594

Total rows: 2830743
Total missing: 5734


In [12]:
processed_counts_2018 = []

for file_path in files_2018:
    output_path = processed_dir_2018 / f"{file_path.stem}.parquet"

    df = preprocess_2018_file(file_path)
    df.to_parquet(output_path, index=False)

    processed_counts_2018.append({
        "file": file_path.name,
        "rows": len(df),
        "missing": df[features_2018].isna().sum().sum()
    })

    del df

processed_counts_2018 = pd.DataFrame(processed_counts_2018)

print(processed_counts_2018)
print()
print("Total rows:", processed_counts_2018["rows"].sum())
print("Total missing:", processed_counts_2018["missing"].sum())

                                                file    rows  missing
0  Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...  771587        0
1  Bruteforce-Wednesday-14-02-2018_TrafficForML_C...  619346        0
2  DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...  954846        0
3  DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...  561396        0
4  DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...  794812        0
5  DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...  591873        0
6  Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...  456873        0
7  Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...  249170        0
8  Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...  830224        0
9  Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...  829405        0

Total rows: 6659532
Total missing: 0


In [13]:
processed_files_2017 = sorted(processed_dir_2017.glob("*.parquet"))
processed_files_2018 = sorted(processed_dir_2018.glob("*.parquet"))

print("2017 processed files:", len(processed_files_2017))
print("2018 processed files:", len(processed_files_2018))

sample_2017_processed = pd.read_parquet(processed_files_2017[0])
sample_2018_processed = pd.read_parquet(processed_files_2018[0])

expected_columns = (
    features_2017_reconciled
    + ["Original_Label", "Target", "Source_File"]
)

print()
print("2017 columns:", len(sample_2017_processed.columns))
print("2018 columns:", len(sample_2018_processed.columns))
print()
print("2017 schema correct:", list(sample_2017_processed.columns) == expected_columns)
print("2018 schema correct:", list(sample_2018_processed.columns) == expected_columns)
print()
print("2017 target values:", sorted(sample_2017_processed["Target"].unique()))
print("2018 target values:", sorted(sample_2018_processed["Target"].unique()))

2017 processed files: 8
2018 processed files: 10

2017 columns: 71
2018 columns: 71

2017 schema correct: True
2018 schema correct: True

2017 target values: [np.int8(0), np.int8(1)]
2018 target values: [np.int8(0), np.int8(1)]
